In [1]:

# 1. Import library yang diperlukan

import requests
import pandas as pd
import json
import os
import time
import hashlib
from datetime import datetime
from zoneinfo import ZoneInfo

print("Library berhasil disiapkan")


Library berhasil disiapkan


In [2]:

# 2. Membuat folder penyimpanan tugas

BASE = "/content/rtm1_2501010333"
RAW = BASE + "/data/raw"
DOCS = BASE + "/docs"

os.makedirs(RAW, exist_ok=True)
os.makedirs(DOCS, exist_ok=True)

print("Folder berhasil dibuat")


Folder berhasil dibuat


In [3]:

# 3. Mengambil data tempat wisata dan budaya di Bali

url = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:180];
(
  nwr["tourism"~"attraction|museum|viewpoint|theme_park|zoo|gallery|artwork"](-9.0,114.4,-8.0,115.75);
  nwr["historic"~"temple|monument|archaeological_site|memorial|fort|ruins|castle"](-9.0,114.4,-8.0,115.75);
  nwr["amenity"="place_of_worship"](-9.0,114.4,-8.0,115.75);
);
out center tags;
"""

# Ganti alamat email dengan email kampus kamu
headers = {
    "User-Agent": "Mahasiswa INSTIKI - Jack Apolinarisa - EMAIL_KAMPUS_KAMU"
}

time.sleep(1)

response = requests.get(
    url,
    params={"data": query},
    headers=headers,
    timeout=240
)

print("Status API:", response.status_code)

response.raise_for_status()

hasil = response.json()

print("Pengambilan data berhasil")
print("Jumlah objek dari API:", len(hasil["elements"]))


Status API: 200
Pengambilan data berhasil
Jumlah objek dari API: 3564


In [4]:

# 4. Menyimpan respons API dalam bentuk JSON

waktu = datetime.now(ZoneInfo("Asia/Makassar"))
tanggal = waktu.strftime("%Y%m%d")

nama_json = f"openstreetmap_bali_{tanggal}.json"
path_json = os.path.join(RAW, nama_json)

with open(path_json, "w", encoding="utf-8") as file:
    json.dump(hasil, file, ensure_ascii=False, indent=2)

print("Data mentah berhasil disimpan:", nama_json)


Data mentah berhasil disimpan: openstreetmap_bali_20261113.json


In [5]:

# 5. Menyusun data menjadi tabel

baris = []

for item in hasil["elements"]:
    tags = item.get("tags", {})

    # Mengambil objek yang memiliki nama
    if not tags.get("name"):
        continue

    if "lat" in item and "lon" in item:
        lat = item["lat"]
        lon = item["lon"]
    elif "center" in item:
        lat = item["center"]["lat"]
        lon = item["center"]["lon"]
    else:
        continue

    kategori = (
        tags.get("tourism")
        or tags.get("historic")
        or tags.get("amenity")
        or "lainnya"
    )

    alamat = tags.get("addr:full", "")
    if not alamat:
        alamat = ", ".join(
            tags[k] for k in ["addr:street", "addr:city"]
            if tags.get(k)
        )

    baris.append({
        "nama_tempat": tags["name"],
        "kategori": kategori,
        "alamat": alamat,
        "latitude": lat,
        "longitude": lon,
        "osm_id": item["id"],
        "jenis_objek": item["type"]
    })

df = pd.DataFrame(baris)

# Menghilangkan objek ganda berdasarkan jenis dan ID
df = df.drop_duplicates(subset=["jenis_objek", "osm_id"])

print("Jumlah data:", len(df))
display(df.head(10))


Jumlah data: 2186


,nama_tempat,kategori,alamat,latitude,longitude,osm_id,jenis_objek
0,Vishnu Statue,artwork,,-8.809807,115.168218,253598001,node
1,Small Beautiful Waterfall,attraction,,-8.175709,115.106360,275573942,node
2,Pura Girikusuma,place_of_worship,,-8.346752,115.483091,314767095,node
3,Pura Besakih,place_of_worship,,-8.375361,115.450706,314905708,node
4,Pura Goa Lawah,place_of_worship,,-8.551525,115.468971,314914957,node
5,Pura Dalem,place_of_worship,,-8.531855,115.507620,316958999,node
6,Taman Sari Temple,place_of_worship,,-8.515673,115.263688,317207547,node
7,Pura,place_of_worship,,-8.510501,115.264636,317207552,node
8,Pura Rambut Siwi,place_of_worship,,-8.403095,114.766122,429235088,node
9,Pura Gede Perancak,place_of_worship,,-8.345327,114.633513,429254842,node


In [6]:

# 6. Memeriksa dan menyimpan data CSV

if len(df) < 100:
    print("Data belum mencapai 100. Perlu pemeriksaan atau perluasan pencarian.")
else:
    print("Data sudah memenuhi minimal 100 record.")

nama_csv = f"openstreetmap_bali_{tanggal}.csv"
path_csv = os.path.join(RAW, nama_csv)

df.to_csv(path_csv, index=False, encoding="utf-8-sig")

print("Nama file:", nama_csv)
print("Jumlah baris:", len(df))
print("Jumlah kolom:", len(df.columns))


Data sudah memenuhi minimal 100 record.
Nama file: openstreetmap_bali_20261113.csv
Jumlah baris: 2186
Jumlah kolom: 7


In [7]:

# 7. Membuat informasi asal data

with open(path_json, "rb") as file:
    isi_file = file.read()

md5_hash = hashlib.md5(isi_file).hexdigest()
ukuran_file = os.path.getsize(path_json)

provenance = {
    "filename": nama_json,
    "csv_filename": nama_csv,
    "file_size_bytes": ukuran_file,
    "md5_hash": md5_hash,
    "source": "OpenStreetMap",
    "url": "https://www.openstreetmap.org/",
    "api_endpoint": url,
    "license": "Open Database License (ODbL)",
    "acquisition_time": waktu.isoformat(),
    "collector_name": "Jack Apolinarisa",
    "collector_nim": "2501010333"
}

path_provenance = os.path.join(DOCS, "provenance.json")

with open(path_provenance, "w", encoding="utf-8") as file:
    json.dump(provenance, file, indent=4, ensure_ascii=False)

print("Provenance berhasil dibuat")
print(provenance)


Provenance berhasil dibuat
{'filename': 'openstreetmap_bali_20261113.json', 'csv_filename': 'openstreetmap_bali_20261113.csv', 'file_size_bytes': 975897, 'md5_hash': 'f524a1d37e900b2fa411a6ab852b933c', 'source': 'OpenStreetMap', 'url': 'https://www.openstreetmap.org/', 'api_endpoint': 'https://overpass-api.de/api/interpreter', 'license': 'Open Database License (ODbL)', 'acquisition_time': '2026-11-13T08:00:52.960201+08:00', 'collector_name': 'Jack Apolinarisa', 'collector_nim': '2501010333'}


In [8]:

# 8. Mengunduh hasil data dan provenance

from google.colab import files

files.download(path_json)
files.download(path_csv)
files.download(path_provenance)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
print("Jumlah record:", len(df))
print("Jumlah kolom:", len(df.columns))
print("Nama kolom:", list(df.columns))
print("Data kosong:")
print(df.isnull().sum())

Jumlah record: 2186
Jumlah kolom: 7
Nama kolom: ['nama_tempat', 'kategori', 'alamat', 'latitude', 'longitude', 'osm_id', 'jenis_objek']
Data kosong:
nama_tempat    0
kategori       0
alamat         0
latitude       0
longitude      0
osm_id         0
jenis_objek    0
dtype: int64
